# Exploração de Dados Públicos Brasileiros

Este notebook realiza a análise exploratória dos dados coletados via pipeline de dados públicos.

## Fontes de Dados
- **IBGE**: Indicadores sociodemográficos e econômicos
- **Receita Federal**: Dados cadastrais de empresas (CNPJ)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Configuração de caminhos
DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Bibliotecas carregadas com sucesso!')

## 1. Exploração de Dados IBGE

In [ ]:
# Carregar dados de estados
estados_path = DATA_DIR / 'ibge' / 'estados.json'
if estados_path.exists():
    df_estados = pd.read_json(estados_path)
    print(f'Total de estados: {len(df_estados)}')
    df_estados.head(10)
else:
    print('Arquivo de estados não encontrado. Execute o coletor primeiro.')

In [ ]:
# Carregar dados de municípios
municipios_path = DATA_DIR / 'ibge' / 'municipios.json'
if municipios_path.exists():
    df_municipios = pd.read_json(municipios_path)
    print(f'Total de municípios: {len(df_municipios)}')
    print(f'\nColunas disponíveis: {list(df_municipios.columns)}')
    df_municipios.head()
else:
    print('Arquivo de municípios não encontrado. Execute o coletor primeiro.')

In [ ]:
# Distribuição de municípios por estado
if 'df_municipios' in dir():
    if 'sigla' in df_municipios.columns:
        contagem_uf = df_municipios['sigla'].value_counts().head(10)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        contagem_uf.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title('Top 10 Estados por Número de Municípios', fontsize=14)
        ax.set_xlabel('UF')
        ax.set_ylabel('Quantidade de Municípios')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        print(f'\nEstatísticas por estado:')
        print(contagem_uf.describe())

In [ ]:
# Análise de municípios por região
if 'df_municipios' in dir() and 'regiao-nome' in df_municipios.columns:
    contagem_regiao = df_municipios['regiao-nome'].value_counts()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    contagem_regiao.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90)
    ax.set_title('Distribuição de Municípios por Região', fontsize=14)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

## 2. Exploração de Dados de CNPJ

In [ ]:
# Carregar dados de CNPJs
import glob

arquivos_cnpj = sorted(glob.glob(str(DATA_DIR / 'cnpj' / 'cnpjs_*.json')))
if arquivos_cnpj:
    arquivo_mais_recente = arquivos_cnpj[-1]
    print(f'Carregando arquivo: {Path(arquivo_mais_recente).name}')
    df_cnpjs = pd.read_json(arquivo_mais_recente)
    print(f'Total de CNPJs consultados: {len(df_cnpjs)}')
    print(f'\nColunas: {list(df_cnpjs.columns)}')
    df_cnpjs.head()
else:
    print('Nenhum arquivo de CNPJ encontrado. Execute o coletor primeiro.')

In [ ]:
# Análise de situação cadastral
if 'df_cnpjs' in dir() and 'situacao_cadastral' in df_cnpjs.columns:
    contagem_situacao = df_cnpjs['situacao_cadastral'].value_counts()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    contagem_situacao.plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Situação Cadastral das Empresas Consultadas', fontsize=14)
    ax.set_xlabel('Quantidade')
    plt.tight_layout()
    plt.show()

In [ ]:
# Análise de capital social
if 'df_cnpjs' in dir() and 'capital_social' in df_cnpjs.columns:
    capital = pd.to_numeric(df_cnpjs['capital_social'], errors='coerce').dropna()
    capital = capital[capital > 0]
    
    if len(capital) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Histograma do capital social
        axes[0].hist(capital, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        axes[0].set_title('Distribuição do Capital Social', fontsize=12)
        axes[0].set_xlabel('Capital Social (R$)')
        axes[0].set_ylabel('Frequência')
        
        # Boxplot por situação cadastral
        if 'situacao_cadastral' in df_cnpjs.columns:
            df_temp = df_cnpjs.copy()
            df_temp['capital_social_num'] = pd.to_numeric(df_temp['capital_social'], errors='coerce')
            df_temp = df_temp.dropna(subset=['capital_social_num'])
            df_temp = df_temp[df_temp['capital_social_num'] > 0]
            
            if len(df_temp) > 0:
                df_temp.boxplot(column='capital_social_num', by='situacao_cadastral', ax=axes[1])
                axes[1].set_title('Capital Social por Situação Cadastral', fontsize=12)
                axes[1].set_xlabel('Situação')
                axes[1].set_ylabel('Capital Social (R$)')
        
        plt.tight_layout()
        plt.show()
        
        print(f'\nEstatísticas do Capital Social:')
        print(f'  Média: R$ {capital.mean():,.2f}')
        print(f'  Mediana: R$ {capital.median():,.2f}')
        print(f'  Máximo: R$ {capital.max():,.2f}')
        print(f'  Mínimo: R$ {capital.min():,.2f}')

## 3. Análise de Atividades Econômicas (CNAE)

In [ ]:
# Top atividades principais
if 'df_cnpjs' in dir() and 'atividade_principal' in df_cnpjs.columns:
    atividades = df_cnpjs['atividade_principal'].value_counts().head(15)
    
    if len(atividades) > 0:
        fig, ax = plt.subplots(figsize=(12, 8))
        atividades.plot(kind='barh', ax=ax, color='coral')
        ax.set_title('Top 15 Atividades Econômicas (CNAE)', fontsize=14)
        ax.set_xlabel('Quantidade de Empresas')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()

## 4. Análise Geográfica

In [ ]:
# Empresas por estado
if 'df_cnpjs' in dir() and 'uf' in df_cnpjs.columns:
    empresas_por_uf = df_cnpjs['uf'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    empresas_por_uf.plot(kind='bar', ax=ax, color='darkblue')
    ax.set_title('Top 10 Estados por Número de Empresas Consultadas', fontsize=14)
    ax.set_xlabel('UF')
    ax.set_ylabel('Quantidade')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Municípios com mais empresas (top 15)
if 'df_cnpjs' in dir() and 'municipio' in df_cnpjs.columns:
    top_municipios = df_cnpjs['municipio'].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    top_municipios.plot(kind='barh', ax=ax, color='darkgreen')
    ax.set_title('Top 15 Municípios por Número de Empresas', fontsize=14)
    ax.set_xlabel('Quantidade de Empresas')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. Resumo dos Dados Coletados

In [ ]:
# Resumo geral
print('=' * 60)
print('RESUMO DOS DADOS PÚBLICOS COLETADOS')
print('=' * 60)

if 'df_estados' in dir():
    print(f'\n📊 IBGE:')
    print(f'   Estados: {len(df_estados)}')

if 'df_municipios' in dir():
    print(f'   Municípios: {len(df_municipios)}')
    if 'sigla' in df_municipios.columns:
        print(f'   Estados com municípios: {df_municipios["sigla"].nunique()}')

if 'df_cnpjs' in dir():
    print(f'\n🏢 CNPJ:')
    print(f'   Empresas consultadas: {len(df_cnpjs)}')
    if 'situacao_cadastral' in df_cnpjs.columns:
        ativas = df_cnpjs[df_cnpjs['situacao_cadastral'] == 'ATIVA'].shape[0]
        print(f'   Empresas ativas: {ativas}')
    if 'uf' in df_cnpjs.columns:
        print(f'   Estados representados: {df_cnpjs["uf"].nunique()}')

print(f'\n{'=' * 60}')